# Construction Cost Prediction — Single Models vs. Two-Stage (SVM + ANN) Models

This notebook implements and compares three modeling approaches for wide-range construction
cost prediction, following the pipeline below:

1. **Single models**: CatBoost, XGBoost, Monotonic GBM (LightGBM with monotonicity
   constraints), and a plain ANN — each trained directly on the full training set (no
   project classification, no transfer learning).
2. **Two-stage models — without transfer learning**: an SVM classifies projects into
   *low-cost* / *high-cost* using a cutoff (Mean, Q3, or Q3+IQR/2); two **independent**
   ANNs are then trained from scratch, one per class.
3. **Two-stage models — with transfer learning**: same SVM classification step, but the
   *high-cost* ANN is initialized from the *low-cost* ANN's weights and fine-tuned, for
   each of the three cutoffs **and** three TL strategies (Conservative, Balanced,
   Aggressive) — 9 combinations total, matching Table 4 in the manuscript.

All experiments are repeated over **10 random seeds** (42–51) for train/test splitting to
report results as `mean ± std`, and evaluated with:

`R²`, `MAPE (%)`, `RMSE`, `RMSE (cost > Q3)`, `RMSE (cost > Q3 + IQR/2)`.


## 0. Setup

Required packages: `pandas`, `numpy`, `scikit-learn`, `tensorflow`, `catboost`, `xgboost`,
`lightgbm`, `xlsxwriter`.

```
pip install pandas numpy scikit-learn tensorflow catboost xgboost lightgbm xlsxwriter
```


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from itertools import product

import tensorflow as tf
from tensorflow.keras.models import Sequential, clone_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.svm import SVC
from sklearn.neural_network import MLPRegressor

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import lightgbm as lgb

tf.keras.backend.clear_session()
pd.set_option("display.width", 140)


## 1. Data preparation

Loads the dataset, one-hot encodes the categorical column `C1`, and prepares the
statistical description used later for cutoff analysis (Mean / Q3 / Q3+IQR/2).


In [ ]:
# ---- EDIT THIS PATH TO YOUR FILE ----
file_path = r'C:\Users\Thu\My Drive\(1) HYU\(3) My Research\(14) M05_with DrK\Data_ML2.xlsx'

df = pd.read_excel(file_path)

X_raw = df.drop(columns=['O'])
y = df['O']

# One-hot encode categorical feature(s)
X = pd.get_dummies(X_raw, columns=['C1'], drop_first=True)
feature_names = X.columns.tolist()

print(f"Dataset shape: {df.shape}")
print(f"Number of features after encoding: {X.shape[1]}")
df.describe()


In [ ]:
# ---- Correlation analysis (feature vs. target) ----
corr_with_target = X.select_dtypes(include=[np.number]).apply(
    lambda col: col.corr(y)
).sort_values(key=np.abs, ascending=False)

print("Correlation of numeric features with target 'O':")
corr_with_target


### Cut-off analysis

Three cutoffs are used to split projects into *low-cost* / *high-cost* segments, and to
define the high-value evaluation segments (`RMSE > Q3`, `RMSE > Q3+IQR/2`):


In [ ]:
Q1, Q3 = y.quantile([0.25, 0.75])
IQR = Q3 - Q1

THRESHOLD_Q3 = Q3
THRESHOLD_Q3_IQR = Q3 + IQR / 2
THRESHOLD_MEAN = y.mean()

CUTOFFS = {
    "Mean": THRESHOLD_MEAN,
    "Q3": THRESHOLD_Q3,
    "Q3+IQR/2": THRESHOLD_Q3_IQR,
}

print("Cutoff / evaluation-segment thresholds:")
for k, v in CUTOFFS.items():
    print(f"  {k:>10s}: {v:,.2f}")


## 2. Metrics

`evaluate_metrics` returns the five reported metrics: `R2`, `MAPE (%)`, `RMSE`,
`RMSE > Q3`, `RMSE > Q3+IQR/2`. The Q3 / Q3+IQR/2 segments are fixed, dataset-wide
thresholds (computed above), independent of which cutoff was used for SVM classification.


In [ ]:
def RMSE(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def MAPE(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100)

def RMSE_segment(y_true, y_pred, threshold):
    mask = y_true >= threshold
    if mask.sum() == 0:
        return np.nan
    return RMSE(y_true[mask], y_pred[mask])

def evaluate_metrics(y_true, y_pred):
    """y_true, y_pred: 1-D numpy arrays, same order/length."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": MAPE(y_true, y_pred),
        "RMSE": RMSE(y_true, y_pred),
        "RMSE > Q3": RMSE_segment(y_true, y_pred, THRESHOLD_Q3),
        "RMSE > Q3+IQR/2": RMSE_segment(y_true, y_pred, THRESHOLD_Q3_IQR),
    }

SEEDS = list(range(42, 52))  # 10 seeds
print(f"Seeds used for all experiments: {SEEDS}")


## 3. ANN builder 

> **Caveat**: unlike CatBoost / XGBoost / Monotonic GBM and the single-model
> ANN, which are tuned with `GridSearchCV`. 

In [ ]:
def build_ann(input_dim, layers, activation, lr):
    tf.keras.backend.clear_session()
    model = Sequential()
    for i, units in enumerate(layers):
        if i == 0:
            model.add(Dense(units, activation=activation, input_dim=input_dim))
        else:
            model.add(Dense(units, activation=activation))
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model


# ---- Fixed ANN hyperparameters (no GridSearchCV) ----
# Picked from the middle of the manuscript's search range:
#   layers in {[64], [128], [128,64]}, lr in {1e-4, 1e-3, 1e-2, 5e-2}, batch in {32, 64}
layers_best = [128, 64]
act_best = 'relu'
lr_best = 1e-3
batch_best = 32

ann_best_params = {
    'layers': layers_best, 'activation': act_best, 'lr': lr_best, 'batch_size': batch_best
}

print("Fixed ANN architecture (used everywhere, no tuning):")
print(f"   Layers: {layers_best}")
print(f"   Activation: {act_best}")
print(f"   Learning rate: {lr_best}")
print(f"   Batch size: {batch_best}")


## 4. Part (1) — Single models: CatBoost, XGBoost, Monotonic GBM, ANN

Each model is trained **directly on the full training set** (70%) and evaluated on the
full test set (30%) — no project classification, no transfer learning. Hyperparameters
for the tree-based models (CatBoost, XGBoost, Monotonic GBM) are searched once (on the
seed=42 split) via `GridSearchCV`, then reused (with a fresh fit per seed) across all 10
seeds.

### Single-model ANN — scikit-learn `MLPRegressor` 

Unlike the Keras-based ANN architecture reused by the two-stage / transfer-learning
models in Sections 5–6 (which needs `clone_model` + layer-freezing for TL), the
**single-model ANN** here uses scikit-learn's `MLPRegressor`, tuned with `GridSearchCV`:

- Search space: `hidden_layer_sizes ∈ {(64,), (128,), (128,64)}`,
  `learning_rate_init ∈ {0.001, 0.01, 0.05}`, `batch_size = 64`
  `early_stopping = False` during the search, `cv = 2`, scored by negative median
  absolute error (searched once on the seed=42 split, reused across all 10 seeds).
- Final per-seed fit: same best architecture/learning rate.


### Monotonic GBM

A LightGBM regressor is used with monotonicity constraints enforced on selected
size/quantity-related features `S = {N3, N4, N6, N8, N10, N16}`, representing project
scale and quality indicators, so predicted cost never decreases as these variables
increase, while unconstrained features keep full modeling flexibility. Features not
present in the dataset are skipped with a warning.


In [ ]:
# ---- Monotone feature set (edit names to match your actual columns if needed) ----
MONOTONE_FEATURES = ['N3', 'N4', 'N6', 'N8', 'N10', 'N16']

missing_monotone = [f for f in MONOTONE_FEATURES if f not in X.columns]
if missing_monotone:
    print(f"WARNING: these monotone features were not found in X.columns and will be "
          f"ignored: {missing_monotone}")

monotone_constraints = [1 if col in MONOTONE_FEATURES else 0 for col in X.columns]
print("Monotone constraint vector (1 = non-decreasing, 0 = unconstrained):")
print(dict(zip(X.columns, monotone_constraints)))


In [ ]:
# ---- Fixed 70/30 split (seed=42) used only for one-time hyperparameter search ----
X_tr42, X_te42, y_tr42, y_te42 = train_test_split(X, y, test_size=0.3, random_state=42)
scaler42 = MinMaxScaler()
X_tr42_s = scaler42.fit_transform(X_tr42)

# ---- One-time hyperparameter search for CatBoost / XGBoost / Monotonic GBM (seed=42) ----

catboost_grid = {'depth': [4, 6, 8], 'learning_rate': [0.03, 0.1], 'iterations': [300, 600]}
xgboost_grid = {'max_depth': [3, 5, 7], 'learning_rate': [0.03, 0.1], 'n_estimators': [300, 600]}
mono_gbm_grid = {'max_depth': [3, 5, 7], 'learning_rate': [0.03, 0.1], 'n_estimators': [300, 600]}

cat_gs = GridSearchCV(
    CatBoostRegressor(verbose=0, random_state=42),
    catboost_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=2
)
cat_gs.fit(X_tr42_s, y_tr42)
catboost_best_params = cat_gs.best_params_
print("Best CatBoost params:", catboost_best_params)

xgb_gs = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0),
    xgboost_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=2
)
xgb_gs.fit(X_tr42_s, y_tr42)
xgboost_best_params = xgb_gs.best_params_
print("Best XGBoost params:", xgboost_best_params)

mono_gs = GridSearchCV(
    lgb.LGBMRegressor(monotone_constraints=monotone_constraints, random_state=42, verbosity=-1),
    mono_gbm_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=2
)
mono_gs.fit(X_tr42_s, y_tr42)
mono_gbm_best_params = mono_gs.best_params_
print("Best Monotonic GBM params:", mono_gbm_best_params)


# ---- Single-model ANN: replaced with scikit-learn's MLPRegressor ----
# (This mirrors an alternative "Simple ANN with GridSearchCV (scikit-learn MLPRegressor)"
#  implementation, kept separate from the Keras `build_ann` architecture used below for
#  the two-stage / transfer-learning ANNs, since TL layer-freezing requires Keras.)
def median_absolute_error(y_true, y_pred):
    """Custom scorer for GridSearchCV."""
    return np.median(np.abs(y_true - y_pred))

ann_mlp_param_grid = {
    'hidden_layer_sizes': [(64,), (128,), (128, 64)],
    'activation': ['relu'],
    'learning_rate_init': [0.001, 0.01, 0.05],
    'batch_size': [64],
    'max_iter': [50],
    'early_stopping': [False],
    'validation_fraction': [0.5],
    'n_iter_no_change': [20],
    'random_state': [42],
}

median_ae_scorer = make_scorer(median_absolute_error, greater_is_better=False)

print("Starting GridSearchCV for the single-model ANN (MLPRegressor)...")
ann_mlp_gs = GridSearchCV(
    estimator=MLPRegressor(solver='adam'),
    param_grid=ann_mlp_param_grid,
    scoring=median_ae_scorer,
    cv=2,
    n_jobs=-1,
    verbose=0,
)
ann_mlp_gs.fit(X_tr42_s, y_tr42)

ann_mlp_best_params = ann_mlp_gs.best_params_
hidden_layers_best = ann_mlp_best_params['hidden_layer_sizes']
mlp_activation_best = ann_mlp_best_params['activation']
mlp_lr_best = ann_mlp_best_params['learning_rate_init']
mlp_batch_best = ann_mlp_best_params['batch_size']

print("Best single-model ANN (MLPRegressor) params:")
print(f"   Hidden layers: {hidden_layers_best}")
print(f"   Activation: {mlp_activation_best}")
print(f"   Learning rate: {mlp_lr_best}")
print(f"   Batch size: {mlp_batch_best}")
print(f"   CV score (median AE): {-ann_mlp_gs.best_score_:.2f}")

ann_mlp_grid_results = pd.DataFrame(ann_mlp_gs.cv_results_)[[
    'param_hidden_layer_sizes', 'param_activation', 'param_learning_rate_init',
    'param_batch_size', 'mean_test_score', 'std_test_score', 'rank_test_score'
]].copy()
ann_mlp_grid_results['mean_test_score'] = -ann_mlp_grid_results['mean_test_score']
ann_mlp_grid_results = ann_mlp_grid_results.sort_values('rank_test_score').reset_index(drop=True)


In [ ]:
# ---- Multi-seed evaluation of single models ----
single_model_results = []

for seed in SEEDS:
    print(f"--- Single models | Seed {seed} ---")

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    y_test_arr = y_test.values

    # ---- CatBoost ----
    cat_model = CatBoostRegressor(**catboost_best_params, verbose=0, random_state=seed)
    cat_model.fit(X_train_s, y_train)
    pred = cat_model.predict(X_test_s)
    m = evaluate_metrics(y_test_arr, pred)
    m.update({'Model': 'CatBoost', 'Seed': seed})
    single_model_results.append(m)

    # ---- XGBoost ----
    xgb_model = XGBRegressor(**xgboost_best_params, random_state=seed, verbosity=0)
    xgb_model.fit(X_train_s, y_train)
    pred = xgb_model.predict(X_test_s)
    m = evaluate_metrics(y_test_arr, pred)
    m.update({'Model': 'XGBoost', 'Seed': seed})
    single_model_results.append(m)

    # ---- Monotonic GBM ----
    mono_model = lgb.LGBMRegressor(
        **mono_gbm_best_params, monotone_constraints=monotone_constraints,
        random_state=seed, verbosity=-1
    )
    mono_model.fit(X_train_s, y_train)
    pred = mono_model.predict(X_test_s)
    m = evaluate_metrics(y_test_arr, pred)
    m.update({'Model': 'Monotonic GBM', 'Seed': seed})
    single_model_results.append(m)

    # ---- ANN (scikit-learn MLPRegressor, no TensorFlow) ----
    ann_model = MLPRegressor(
        hidden_layer_sizes=hidden_layers_best,
        activation=mlp_activation_best,
        learning_rate_init=mlp_lr_best,
        batch_size=mlp_batch_best,
        max_iter=100,
        early_stopping=True,
        validation_fraction=0.5,
        n_iter_no_change=40,
        random_state=seed,
        solver='adam',
        verbose=False,
    )
    ann_model.fit(X_train_s, y_train)
    pred = ann_model.predict(X_test_s)
    m = evaluate_metrics(y_test_arr, pred)
    m.update({'Model': 'ANN', 'Seed': seed})
    single_model_results.append(m)
    del ann_model

df_single = pd.DataFrame(single_model_results)
df_single


In [ ]:
def summarize(df, group_cols, metric_cols=('R2', 'MAPE (%)', 'RMSE', 'RMSE > Q3', 'RMSE > Q3+IQR/2')):
    """Return a mean +/- std summary table, one row per group."""
    g = df.groupby(group_cols)[list(metric_cols)]
    mean_df = g.mean()
    std_df = g.std()
    out = mean_df.copy()
    for col in metric_cols:
        out[col] = [f"{m:.4f} +/- {s:.4f}" if col == 'R2' else f"{m:,.2f} +/- {s:,.2f}"
                    for m, s in zip(mean_df[col], std_df[col])]
    return out.reset_index()

summary_single = summarize(df_single, group_cols=['Model'])
summary_single = summary_single.set_index('Model').loc[['CatBoost', 'XGBoost', 'Monotonic GBM', 'ANN']].reset_index()
summary_single


## 5. Part (2) — Two-stage models WITHOUT transfer learning

For each of the three cutoffs (Mean, Q3, Q3+IQR/2):

1. An **SVM classifier** labels each training project as low-cost (0) / high-cost (1)
   using that cutoff.
2. Two **independent** ANNs are trained from scratch — one on the low-cost subset, one on
   the high-cost subset — using the shared architecture found in Section 3. There is no
   weight sharing between the two ANNs.
3. At test time, the SVM assigns each test project to a class, and the corresponding ANN
   produces the cost prediction.


In [ ]:
# ---- SVM search range as reported in the manuscript (Figure 4) ----
svm_grid = {
    'C': [0.1, 1, 10, 50, 100],
    'kernel': ['rbf', 'linear'],
    'gamma': ['auto'],
}

def fit_svm_classifier(X_train_s, y_train, cutoff):
    y_train_cls = (y_train >= cutoff).astype(int)
    svm_gs = GridSearchCV(SVC(), svm_grid, cv=5, scoring='f1', n_jobs=2)
    svm_gs.fit(X_train_s, y_train_cls)
    return svm_gs.best_estimator_, svm_gs.best_params_, svm_gs.best_score_, y_train_cls


In [ ]:
two_stage_notransfer_results = []
svm_logs_notransfer = []

for cutoff_name, cutoff_value in CUTOFFS.items():
    for seed in SEEDS:
        print(f"--- Two-stage (no TL) | Cutoff={cutoff_name} | Seed={seed} ---")

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
        scaler = MinMaxScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        y_test_arr = y_test.values

        svm, svm_params, svm_f1, train_cls = fit_svm_classifier(X_train_s, y_train, cutoff_value)
        test_cls = svm.predict(X_test_s)

        svm_logs_notransfer.append({
            'Cutoff': cutoff_name, 'Seed': seed,
            'C': svm_params['C'], 'Gamma': svm_params['gamma'], 'CV_f1': svm_f1
        })

        # ---- Low-cost ANN (trained from scratch) ----
        low_ann = build_ann(X_train_s.shape[1], layers_best, act_best, lr_best)
        if np.any(train_cls == 0):
            low_ann.fit(
                X_train_s[train_cls == 0], y_train[train_cls == 0],
                epochs=500, batch_size=batch_best, verbose=0,
                callbacks=[EarlyStopping(patience=40, restore_best_weights=True)]
            )

        # ---- High-cost ANN (trained from scratch, fully independent) ----
        high_ann = build_ann(X_train_s.shape[1], layers_best, act_best, lr_best)
        if np.any(train_cls == 1):
            high_ann.fit(
                X_train_s[train_cls == 1], y_train[train_cls == 1],
                epochs=500, batch_size=batch_best, verbose=0,
                callbacks=[EarlyStopping(patience=40, restore_best_weights=True)]
            )

        # ---- Combined prediction ----
        y_pred = np.zeros(len(y_test))
        if np.any(test_cls == 0):
            y_pred[test_cls == 0] = low_ann.predict(X_test_s[test_cls == 0], verbose=0).ravel()
        if np.any(test_cls == 1):
            y_pred[test_cls == 1] = high_ann.predict(X_test_s[test_cls == 1], verbose=0).ravel()

        m = evaluate_metrics(y_test_arr, y_pred)
        m.update({'Cutoff': cutoff_name, 'Seed': seed})
        two_stage_notransfer_results.append(m)

        del low_ann, high_ann
        tf.keras.backend.clear_session()

df_two_stage_notransfer = pd.DataFrame(two_stage_notransfer_results)
df_two_stage_notransfer


In [ ]:
summary_two_stage_notransfer = summarize(df_two_stage_notransfer, group_cols=['Cutoff'])
summary_two_stage_notransfer = (
    summary_two_stage_notransfer.set_index('Cutoff').loc[list(CUTOFFS.keys())].reset_index()
)
summary_two_stage_notransfer


## 6. Part (3) — Two-stage models WITH transfer learning

Same SVM classification step as Part (2), but now the *high-cost* ANN is **not** trained
from scratch: it is cloned from the trained *low-cost* ANN and fine-tuned on the
high-cost subset. Three TL strategies from the manuscript are evaluated, for each of the
three cutoffs (9 combinations total):

| Strategy | Description | Frozen layers | Fine-tune LR |
|---|---|---|---|
| Conservative | most hidden layers frozen, small LR | all but the last layer | 1e-3 |
| Balanced | partial freezing, moderate LR | all but the last two layers | 1e-2 |
| Aggressive | all layers trainable, full adaptation | none | 5e-2 |

The base (low-cost) ANN is trained once per (cutoff, seed) and reused as the starting
point for all three strategies, so each strategy differs only in which layers are frozen
and the fine-tuning learning rate.

In [ ]:
# ---- Three TL strategies as reported in the manuscript ----
# Conservative: most hidden layers frozen, small learning rate -> only last layer trainable
# Balanced:     partial freezing, moderate learning rate       -> last two layers trainable
# Aggressive:   all layers trainable, full adaptation          -> nothing frozen
TRANSFER_STRATEGIES = {
    "Conservative": {"freeze": -1, "lr": 1e-3, "epochs": 500},
    "Balanced":     {"freeze": -2, "lr": 1e-2, "epochs": 500},
    "Aggressive":   {"freeze": 0,  "lr": 5e-2, "epochs": 500},
}

two_stage_transfer_results = []
svm_logs_transfer = []

for cutoff_name, cutoff_value in CUTOFFS.items():
    for seed in SEEDS:
        print(f"--- Two-stage (with TL) | Cutoff={cutoff_name} | Seed={seed} ---")

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)
        scaler = MinMaxScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        y_test_arr = y_test.values

        svm, svm_params, svm_f1, train_cls = fit_svm_classifier(X_train_s, y_train, cutoff_value)
        test_cls = svm.predict(X_test_s)

        svm_logs_transfer.append({
            'Cutoff': cutoff_name, 'Seed': seed,
            'C': svm_params['C'], 'Kernel': svm_params['kernel'],
            'Gamma': svm_params['gamma'], 'CV_f1': svm_f1
        })

        # ---- Base (low-cost) ANN: trained once per (cutoff, seed), reused by every strategy ----
        base_ann = build_ann(X_train_s.shape[1], layers_best, act_best, lr_best)
        if np.any(train_cls == 0):
            base_ann.fit(
                X_train_s[train_cls == 0], y_train[train_cls == 0],
                epochs=500, batch_size=batch_best, verbose=0,
                callbacks=[EarlyStopping(patience=40, restore_best_weights=True)]
            )
        base_weights = base_ann.get_weights()

        # ---- Transferred (high-cost) ANN, for each TL strategy ----
        for strategy_name, cfg in TRANSFER_STRATEGIES.items():
            tl_model = clone_model(base_ann)
            tl_model.set_weights(base_weights)

            freeze_upto = cfg["freeze"]
            if freeze_upto != 0:
                for layer in tl_model.layers[:freeze_upto]:
                    layer.trainable = False
            tl_model.compile(optimizer=Adam(learning_rate=cfg["lr"]), loss='mse')

            if np.any(train_cls == 1):
                tl_model.fit(
                    X_train_s[train_cls == 1], y_train[train_cls == 1],
                    epochs=cfg["epochs"], batch_size=batch_best, verbose=0,
                    callbacks=[EarlyStopping(patience=40, restore_best_weights=True)]
                )

            # ---- Combined prediction ----
            y_pred = np.zeros(len(y_test))
            if np.any(test_cls == 0):
                y_pred[test_cls == 0] = base_ann.predict(X_test_s[test_cls == 0], verbose=0).ravel()
            if np.any(test_cls == 1):
                y_pred[test_cls == 1] = tl_model.predict(X_test_s[test_cls == 1], verbose=0).ravel()

            m = evaluate_metrics(y_test_arr, y_pred)
            m.update({'Cutoff': cutoff_name, 'Seed': seed, 'Strategy': strategy_name})
            two_stage_transfer_results.append(m)

            del tl_model
            tf.keras.backend.clear_session()

        del base_ann

df_two_stage_transfer = pd.DataFrame(two_stage_transfer_results)
df_two_stage_transfer


In [ ]:
# Order rows to match Table 4: cutoff (Mean, Q3, Q3+IQR/2) x strategy (Conservative, Balanced, Aggressive)
summary_two_stage_transfer = summarize(df_two_stage_transfer, group_cols=['Cutoff', 'Strategy'])

cutoff_order = list(CUTOFFS.keys())
strategy_order = list(TRANSFER_STRATEGIES.keys())
summary_two_stage_transfer['Cutoff'] = pd.Categorical(
    summary_two_stage_transfer['Cutoff'], categories=cutoff_order, ordered=True
)
summary_two_stage_transfer['Strategy'] = pd.Categorical(
    summary_two_stage_transfer['Strategy'], categories=strategy_order, ordered=True
)
summary_two_stage_transfer = (
    summary_two_stage_transfer.sort_values(['Cutoff', 'Strategy']).reset_index(drop=True)
)
summary_two_stage_transfer


## 7. Combined comparison table (Table-4 style)

Reproduces the layout of *Table 4. Performance of single and two-stage models*: single
models first, then two-stage-no-transfer by cutoff, then two-stage-with-transfer by
cutoff.


In [ ]:
combined_rows = []

for _, row in summary_single.iterrows():
    combined_rows.append({
        'Model': row['Model'], 'Cutoff': '', 'Transfer strategy': '',
        'R2': row['R2'], 'MAPE (%)': row['MAPE (%)'], 'RMSE': row['RMSE'],
        'RMSE > Q3': row['RMSE > Q3'], 'RMSE > Q3+IQR/2': row['RMSE > Q3+IQR/2'],
    })

for _, row in summary_two_stage_notransfer.iterrows():
    combined_rows.append({
        'Model': 'Two-stage, no transfer', 'Cutoff': row['Cutoff'], 'Transfer strategy': '',
        'R2': row['R2'], 'MAPE (%)': row['MAPE (%)'], 'RMSE': row['RMSE'],
        'RMSE > Q3': row['RMSE > Q3'], 'RMSE > Q3+IQR/2': row['RMSE > Q3+IQR/2'],
    })

for _, row in summary_two_stage_transfer.iterrows():
    combined_rows.append({
        'Model': 'Two-stage, transfer', 'Cutoff': row['Cutoff'], 'Transfer strategy': row['Strategy'],
        'R2': row['R2'], 'MAPE (%)': row['MAPE (%)'], 'RMSE': row['RMSE'],
        'RMSE > Q3': row['RMSE > Q3'], 'RMSE > Q3+IQR/2': row['RMSE > Q3+IQR/2'],
    })

table4_style = pd.DataFrame(combined_rows)
table4_style


## 8. Export results to Excel


In [ ]:
with pd.ExcelWriter("Results_Summary.xlsx", engine="xlsxwriter") as writer:
    # Raw per-seed results
    df_single.to_excel(writer, sheet_name="Single_Models_raw", index=False)
    df_two_stage_notransfer.to_excel(writer, sheet_name="TwoStage_NoTL_raw", index=False)
    df_two_stage_transfer.to_excel(writer, sheet_name="TwoStage_TL_raw", index=False)

    # Mean +/- SD summaries
    summary_single.to_excel(writer, sheet_name="Single_Models_summary", index=False)
    summary_two_stage_notransfer.to_excel(writer, sheet_name="TwoStage_NoTL_summary", index=False)
    summary_two_stage_transfer.to_excel(writer, sheet_name="TwoStage_TL_summary", index=False)

    # Table-4-style combined view
    table4_style.to_excel(writer, sheet_name="Combined_Table4_style", index=False)

    # Supporting logs
    pd.DataFrame(svm_logs_notransfer).to_excel(writer, sheet_name="SVM_Logs_NoTL", index=False)
    pd.DataFrame(svm_logs_transfer).to_excel(writer, sheet_name="SVM_Logs_TL", index=False)
    pd.DataFrame([ann_best_params]).to_excel(writer, sheet_name="Best_ANN_Keras_Params", index=False)
    ann_mlp_grid_results.to_excel(writer, sheet_name="ANN_MLP_GridSearch", index=False)
    pd.DataFrame([ann_mlp_best_params]).to_excel(writer, sheet_name="Best_ANN_MLP_Params", index=False)
    pd.DataFrame([catboost_best_params]).to_excel(writer, sheet_name="Best_CatBoost_Params", index=False)
    pd.DataFrame([xgboost_best_params]).to_excel(writer, sheet_name="Best_XGBoost_Params", index=False)
    pd.DataFrame([mono_gbm_best_params]).to_excel(writer, sheet_name="Best_MonoGBM_Params", index=False)
    pd.DataFrame({
        'Cutoff': list(CUTOFFS.keys()),
        'Value': list(CUTOFFS.values()),
    }).to_excel(writer, sheet_name="Thresholds", index=False)

print("All results exported to 'Results_Summary.xlsx':")
print("  - Single_Models_raw / _summary")
print("  - TwoStage_NoTL_raw / _summary")
print("  - TwoStage_TL_raw / _summary")
print("  - Combined_Table4_style")
print("  - SVM logs, ANN (MLPRegressor) grid search, best hyperparameters, thresholds")
